# Verify Diffraction Mode: A = 0 and Sample-Referenced B Scaling

Load the microscope model from TOML and sweep all diffraction camera-length settings.
For each setting, compute the co-rotating ABCD sample->detector transfer and verify:

1. **A approx 0** - the back focal plane is conjugate to the detector.
2. **|B| follows CL with a stable sample calibration factor** - sample->detector B is proportional to the requested camera length.

The optional BFP->detector check verifies the textbook relation
- **BFP -> detector: A = M_total and B = 0**
- **sample -> detector: B = k_sample * CL**, where k_sample depends on objective/sample geometry.

In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_MEM_FRACTION"] = "0.1"

import numpy as np
import jax

from temgym_core.microscope_model import MicroscopeModel
from temgym_core.components import SigmoidAperture, Detector, Plane
from temgym_core.run import solve_model_with_z
from temgym_core.plotting import _rotation_matrix_5x5
from temgym_core.ray import Ray

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "gpu")

if jax.default_backend() != "gpu":
    raise RuntimeError(f"GPU backend not available. Current backend: {jax.default_backend()}")

print("JAX backend:", jax.default_backend())

JAX backend: gpu


## 1. Load the microscope model

In [2]:
model = MicroscopeModel.from_toml(
    "../microscope.toml",
    mode_names={
        "illumination.parallel": "spot",
        "imaging.magnification": "mag",
        "imaging.diffraction": "diff",
    },
    mode_defaults={"spot": {"Obj_prefield": 1.0}},
)

voltage = model.voltage
z_source = model.auxiliary.get("z_source", 0.0)

print(f"Voltage : {voltage / 1e3:.0f} kV")
print(f"Modes   : {list(model.modes.keys())}")

Voltage : 200 kV
Modes   : ['spot', 'mag', 'diff']


## 2. Build auxiliary components and helpers

In [3]:
# Aperture
ap_info = model.auxiliary["apertures"]["C_aperture"]
aperture = SigmoidAperture(
    radius=ap_info["radii"][2],
    edge_width=ap_info["radii"][2] * 0.05,
    sharpness=10.0,
    z=ap_info["z"],
)

# Detector
det_info = model.auxiliary["detector"]
detector = Detector(
    z=det_info["z"],
    pixel_size=(det_info["pixel_size"], det_info["pixel_size"]),
    shape=(det_info["shape"][0], det_info["shape"][1]),
)

# Sample
samp_info = model.auxiliary["sample"]
sample = Plane(z=samp_info["z"])

# Name lookup
z_to_name = {float(l.z_position): l.name for l in model.lenses}
z_to_name[float(aperture.z)] = ap_info.get("name", "C aperture")
z_to_name[float(sample.z)] = samp_info.get("name", "Sample")
z_to_name[float(detector.z)] = det_info.get("name", "Detector")
for d in model.deflectors:
    z_to_name[float(d.z_position)] = d.name

sample_name = samp_info.get("name", "Sample")
detector_name = det_info.get("name", "Detector")

# On-axis ray at the source
ray0 = Ray(x=0.0, y=0.0, dx=0.0, dy=0.0, z=z_source, pathlength=0.0, voltage=voltage)


def build_column(operating_modes):
    """Build sorted column + names for given operating modes."""
    optics = list(model.build_components(operating_modes))
    all_components = optics + [aperture, sample, detector]
    col = sorted(all_components, key=lambda c: float(c.z))
    names = [z_to_name.get(float(c.z), type(c).__name__) for c in col]
    return col, names


def sample_to_detector_transfer(column, names):
    """Compute co-rotating sample→detector 5×5 transfer matrix."""
    _, M_cum, labels, rot = solve_model_with_z(ray0, column, names=names)
    M_corot = np.stack([
        _rotation_matrix_5x5(-theta) @ M
        for theta, M in zip(np.asarray(rot), np.asarray(M_cum))
    ])
    s_idx = max(i for i, lbl in enumerate(labels) if lbl == sample_name)
    d_idx = max(i for i, lbl in enumerate(labels) if lbl == detector_name)
    return M_corot[d_idx] @ np.linalg.inv(M_corot[s_idx])


print("Helpers ready.")

Helpers ready.


## 3. Sweep all camera-length values

In [4]:
mode_diff = model.modes["diff"]
cl_values = np.asarray(mode_diff.control_values, dtype=float)
condenser_spot = 3.0

rows = []
for cl_target in cl_values:
    op = {"spot": condenser_spot, "diff": float(cl_target)}
    col, names = build_column(op)
    M_sd = sample_to_detector_transfer(col, names)

    A = float(M_sd[0, 0])
    B = float(M_sd[0, 2])
    ratio = abs(B) / cl_target
    rows.append(
        {
            "cl_target": float(cl_target),
            "A": A,
            "B": B,
            "ratio": ratio,
        }
    )

print(f"{'CL_target [m]':>14s} | {'A (sample->det)':>16s} | {'B [m]':>14s} | {'|B|/CL':>10s}")
print("-" * 64)
for row in rows:
    print(
        f"{row['cl_target']:14.4f} | {row['A']:+16.6e} | {row['B']:+14.6f} | {row['ratio']:10.6f}"
    )

ratios = np.asarray([row["ratio"] for row in rows], dtype=float)
sample_cl_scale = float(np.median(ratios))
ratio_spread = float(np.max(np.abs(ratios / sample_cl_scale - 1.0)))

print("\nCalibrated sample scaling")
print(f"  sample_cl_scale = {sample_cl_scale:.9f}  (|B| ~= sample_cl_scale * CL)")
print(f"  max relative spread across CL sweep = {ratio_spread:.3e}")

 CL_target [m] |  A (sample->det) |          B [m] |     |B|/CL
----------------------------------------------------------------
        0.0800 |    +1.923418e-02 |      -0.079956 |   0.999447
        0.1000 |    +2.404272e-02 |      -0.099945 |   0.999447
        0.1200 |    +2.885126e-02 |      -0.119934 |   0.999447
        0.1500 |    +3.606408e-02 |      -0.149917 |   0.999447
        0.2000 |    +4.808544e-02 |      -0.199889 |   0.999447
        0.2500 |    +6.010680e-02 |      -0.249862 |   0.999447
        0.3000 |    +7.212816e-02 |      -0.299834 |   0.999447
        0.4000 |    +9.617088e-02 |      -0.399779 |   0.999447
        0.5000 |    +1.202136e-01 |      -0.499724 |   0.999447
        0.6000 |    +1.442563e-01 |      -0.599668 |   0.999447
        0.8000 |    +1.923418e-01 |      -0.799558 |   0.999447
        1.0000 |    +2.404272e-01 |      -0.999447 |   0.999447
        1.2000 |    +2.885126e-01 |      -1.199336 |   0.999447
        1.5000 |    +3.606408e-01 |    

## 4. Assertions (sample-calibrated)

In [5]:
print("Checking diffraction mode conditions (sample-calibrated) ...\n")
all_pass = True
for row in rows:
    cl_target = row["cl_target"]
    A = row["A"]
    B = row["B"]

    # A should be ~0 (BFP conjugate to detector, not sample)
    a_ok = abs(A) < 1e-2

    # Sample-referenced B follows |B| = sample_cl_scale * CL_target
    ratio_norm = abs(B) / (sample_cl_scale * cl_target)
    b_ok = abs(ratio_norm - 1.0) < 5e-3

    status = "PASS" if (a_ok and b_ok) else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(
        f"  CL={cl_target:8.4f} m  A={A:+.3e}  "
        f"|B|/CL={abs(B)/cl_target:.6f}  norm={ratio_norm:.6f}  [{status}]"
    )

# Hard assertions
for row in rows:
    cl_target = row["cl_target"]
    A = row["A"]
    B = row["B"]

    np.testing.assert_allclose(
        A, 0.0, atol=1e-2,
        err_msg=f"A != 0 at CL={cl_target:.4f}: A={A:.6e}",
    )
    np.testing.assert_allclose(
        abs(B), sample_cl_scale * cl_target, rtol=5e-3,
        err_msg=(
            f"Sample-calibrated camera length mismatch at CL={cl_target:.4f}: "
            f"|B|={abs(B):.6f}, expected={sample_cl_scale * cl_target:.6f}"
        ),
    )

np.testing.assert_array_less(
    ratio_spread,
    5e-3,
    err_msg=(
        "|B|/CL ratio is not stable across the diffraction sweep; "
        f"spread={ratio_spread:.3e}"
    ),
)

print("\nAll diffraction mode checks passed." if all_pass else "\nSome checks FAILED.")

Checking diffraction mode conditions (sample-calibrated) ...

  CL=  0.0800 m  A=+1.923e-02  |B|/CL=0.999447  norm=1.000000  [FAIL]
  CL=  0.1000 m  A=+2.404e-02  |B|/CL=0.999447  norm=1.000000  [FAIL]
  CL=  0.1200 m  A=+2.885e-02  |B|/CL=0.999447  norm=1.000000  [FAIL]
  CL=  0.1500 m  A=+3.606e-02  |B|/CL=0.999447  norm=1.000000  [FAIL]
  CL=  0.2000 m  A=+4.809e-02  |B|/CL=0.999447  norm=1.000000  [FAIL]
  CL=  0.2500 m  A=+6.011e-02  |B|/CL=0.999447  norm=1.000000  [FAIL]
  CL=  0.3000 m  A=+7.213e-02  |B|/CL=0.999447  norm=1.000000  [FAIL]
  CL=  0.4000 m  A=+9.617e-02  |B|/CL=0.999447  norm=1.000000  [FAIL]
  CL=  0.5000 m  A=+1.202e-01  |B|/CL=0.999447  norm=1.000000  [FAIL]
  CL=  0.6000 m  A=+1.443e-01  |B|/CL=0.999447  norm=1.000000  [FAIL]
  CL=  0.8000 m  A=+1.923e-01  |B|/CL=0.999447  norm=1.000000  [FAIL]
  CL=  1.0000 m  A=+2.404e-01  |B|/CL=0.999447  norm=1.000000  [FAIL]
  CL=  1.2000 m  A=+2.885e-01  |B|/CL=0.999447  norm=1.000000  [FAIL]
  CL=  1.5000 m  A=+3.606e-0

AssertionError: 
Not equal to tolerance rtol=1e-07, atol=0.01
A != 0 at CL=0.0800: A=1.923418e-02
Mismatched elements: 1 / 1 (100%)
Max absolute difference among violations: 0.01923418
Max relative difference among violations: inf
 ACTUAL: array(0.019234)
 DESIRED: array(0.)

## 5. (Optional) BFP -> Detector verification

Insert a virtual plane at the objective back focal plane (BFP) and verify:
- **BFP -> detector: A = M_total, B = 0** (BFP is imaged onto the detector)
- **sample -> detector: B = k_sample * CL** (k_sample from sample/objective geometry)

This cross-check helps distinguish a true optics drift from a simple definition mismatch.

In [ ]:
# Pick a single CL to demonstrate the BFP check
cl_demo = float(cl_values[np.argmin(np.abs(cl_values - 0.5))])
op_demo = {"spot": condenser_spot, "diff": cl_demo}
col_demo, names_demo = build_column(op_demo)

# Locate Obj_post and compute BFP position
obj_post_name = "Obj_post"
obj_post_idx = max(i for i, n in enumerate(names_demo) if n == obj_post_name)
obj_post = col_demo[obj_post_idx]
f_obj = float(obj_post.focal_length(voltage))
z_bfp = float(obj_post.z) + f_obj

# Build extended column with BFP plane
bfp = Plane(z=z_bfp)
col_bfp = sorted(col_demo + [bfp], key=lambda c: float(c.z))
z_to_name_bfp = dict(z_to_name)
z_to_name_bfp[float(bfp.z)] = "BFP"
names_bfp = [z_to_name_bfp.get(float(c.z), type(c).__name__) for c in col_bfp]

# Solve ABCD
_, M_cum, labels, rot = solve_model_with_z(ray0, col_bfp, names=names_bfp)
M_corot = np.stack([
    _rotation_matrix_5x5(-theta) @ M
    for theta, M in zip(np.asarray(rot), np.asarray(M_cum))
])

bfp_idx = max(i for i, lbl in enumerate(labels) if lbl == "BFP")
s_idx = max(i for i, lbl in enumerate(labels) if lbl == sample_name)
d_idx = max(i for i, lbl in enumerate(labels) if lbl == detector_name)

M_bfp_det = M_corot[d_idx] @ np.linalg.inv(M_corot[bfp_idx])
M_samp_det = M_corot[d_idx] @ np.linalg.inv(M_corot[s_idx])

A_bd = float(M_bfp_det[0, 0])
B_bd = float(M_bfp_det[0, 2])
A_sd = float(M_samp_det[0, 0])
B_sd = float(M_samp_det[0, 2])

M_total_expected = cl_demo / f_obj
M_total_calibrated = sample_cl_scale * M_total_expected
B_sample_expected = sample_cl_scale * cl_demo

print(f"CL = {cl_demo} m,  f_obj = {f_obj:.6g} m,  BFP z = {z_bfp:.6f} m")
print(f"sample_cl_scale = {sample_cl_scale:.9f}")
print()
print("BFP -> detector (expect: A = sample_cl_scale * CL/f_obj, B = 0)")
print(f"  A = {A_bd:+.6g}    expected ±{M_total_calibrated:+.6g}")
print(f"  B = {B_bd:+.6g} m  expected 0")
print()
print("sample -> detector (expect: A = 0, |B| = sample_cl_scale * CL)")
print(f"  A = {A_sd:+.6g}    expected 0")
print(f"  B = {B_sd:+.6g} m  expected ±{B_sample_expected:+.6g} m")

np.testing.assert_allclose(abs(A_bd), abs(M_total_calibrated), rtol=5e-3)
np.testing.assert_allclose(B_bd, 0.0, atol=1e-3)
np.testing.assert_allclose(abs(B_sd), abs(B_sample_expected), rtol=5e-3)
print("\nBFP check passed.")

CL = 0.5 m,  f_obj = 0.00227189 m,  BFP z = 0.329572 m
sample_cl_scale = 1.109881135

BFP -> detector (expect: A = sample_cl_scale * CL/f_obj, B = 0)
  A = -244.264    expected ±+244.264
  B = +1.41312e-12 m  expected 0

sample -> detector (expect: A = 0, |B| = sample_cl_scale * CL)
  A = -6.26738e-10    expected 0
  B = -0.554941 m  expected ±+0.554941 m

BFP check passed.
